# Minimal cross-store query — Netherlands 2024

Pull **CO2** from the ICOS Obspack zarr, **monthly NEE** from the
FLUXNET zarr, and **fCO₂** from the ICOS Ocean (SOCAT) zarr for
stations / cruises inside a Netherlands bounding box, restricted to
2024.

- **Region**: lat 50.7–53.6, lon 3.3–7.3
- **Time**: 2024-01-01 → 2024-12-31

Each store has a *combined-view group* (built by `combine_to_dim.py`)
where the per-station / per-cruise data shares one indexed dimension,
so spatial and temporal selection becomes a single xarray expression.

| Store | Combined group | Index dim | Spatial coords |
|---|---|---|---|
| `icos-obspack.zarr` | `co2`, `ch4`, `n2o`, `co` | `station` | `lat(station)`, `lon(station)`, `intake_height(station)` |
| `icos-fluxnet.zarr` | `_combined/fluxnet_dd` … `_yy` | `station` | `lat(station)`, `lon(station)`, `station_elevation(station)` |
| `icos-socat.zarr`   | `_combined`               | `deployment` | `lat(deployment, time)`, `lon(deployment, time)` (SOCAT lat/lon vary along the cruise) |

The SOCAT combined view is built by `combine_to_dim.py socat`; QC-failed
rows (WOCE flag > 2 on the variable's QC or on the position QC) are
NaN-padded at build time, so consumers don't need to re-mask.

In [8]:
import xarray as xr

LAT_MIN, LAT_MAX = 50.7, 53.6
LON_MIN, LON_MAX = 3.3, 7.3
T0, T1 = "2024-01-01", "2024-12-31"

## Obspack — CO2

In [9]:
ds = xr.open_zarr("zarr//icos-obspack.zarr", group="co2", consolidated=False)

co2_nl = (
    ds["co2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time_co2=slice(T0, T1))
)

# Per-station summary — coords travel with the DataArray
for sid in co2_nl.station.values:
    da = co2_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(co2_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(co2_nl.lon.sel(station=sid)):.3f}  "
          f"intake_height={float(co2_nl.intake_height.sel(station=sid)):>4.0f} m  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.2f} ppm")

CBW127   lat=51.970 lon=4.926  intake_height= 127 m  n=8549  mean=432.71 ppm
CBW207   lat=51.970 lon=4.926  intake_height= 207 m  n=8739  mean=430.82 ppm
CBW27    lat=51.970 lon=4.926  intake_height=  27 m  n=8526  mean=439.16 ppm
CBW67    lat=51.970 lon=4.926  intake_height=  67 m  n=8541  mean=435.18 ppm
JUE120   lat=50.910 lon=6.410  intake_height= 120 m  n=8443  mean=434.02 ppm
JUE50    lat=50.910 lon=6.410  intake_height=  50 m  n=7996  mean=436.78 ppm
JUE80    lat=50.910 lon=6.410  intake_height=  80 m  n=7992  mean=435.31 ppm
LUT0     lat=53.404 lon=6.353  intake_height=  60 m  n=8332  mean=431.81 ppm


## FLUXNET — monthly NEE

Selects the VUT/REF NEE variant from each site's `fluxnet_mm` sub-group.

In [10]:
ds = xr.open_zarr("zarr//icos-fluxnet.zarr", group="_combined/fluxnet_mm", consolidated=False)

nee_nl = (
    ds["NEE"]
      .sel(ustar_threshold="VUT", nee_variant="REF")
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["NEE"].attrs.get("units", "")
for sid in nee_nl.station.values:
    da = nee_nl.sel(station=sid)
    print(f"{sid:8s} lat={float(nee_nl.lat.sel(station=sid)):.3f} "
          f"lon={float(nee_nl.lon.sel(station=sid)):.3f}  "
          f"n={int(da.count())}  "
          f"mean={float(da.mean()):.3f} {units}")

BE-Bra   lat=51.308 lon=4.520  n=12  mean=-0.735 umol m-2 s-1
BE-Maa   lat=50.980 lon=5.632  n=12  mean=-0.095 umol m-2 s-1
DE-RuS   lat=50.866 lon=6.447  n=12  mean=-0.562 umol m-2 s-1
NL-Loo   lat=52.166 lon=5.744  n=12  mean=-0.261 umol m-2 s-1


## SOCAT — fCO₂

Same pattern as the other two stores: open the `_combined` group, mask
on `lat`/`lon` (which are 2-D `(deployment, time)` here because SOCAT
lat/lon vary along each cruise — xarray broadcasts the mask correctly),
slice on `time`.

In [11]:
import numpy as np

ds = xr.open_zarr("zarr//icos-socat.zarr", group="_combined", consolidated=False)

fco2_nl = (
    ds["fCO2"]
      .where(
          (ds.lat >= LAT_MIN) & (ds.lat <= LAT_MAX) &
          (ds.lon >= LON_MIN) & (ds.lon <= LON_MAX),
          drop=True,
      )
      .sel(time=slice(T0, T1))
)

units = ds["fCO2"].attrs.get("units", "µatm")
valid = fco2_nl.values[np.isfinite(fco2_nl.values)]
print(f"SOCAT fCO2 — {fco2_nl.sizes.get('deployment', 0)} contributing deployments × "
      f"{fco2_nl.sizes.get('time', 0)} timestamps  →  "
      f"{valid.size} QC-passing samples")

if valid.size:
    print(f"\npooled mean fCO2 = {valid.mean():.1f} {units}  "
          f"(range {valid.min():.1f}–{valid.max():.1f})")
    # Per-deployment summary
    for dep in fco2_nl.deployment.values:
        da = fco2_nl.sel(deployment=dep)
        v = da.values[np.isfinite(da.values)]
        if not v.size:
            continue
        station = str(ds.station_id.sel(deployment=dep).values)
        print(f"  {str(dep):14s}  {station:30s}  n={v.size:>5d}  mean={v.mean():.1f} {units}")

SOCAT fCO2 — 7 contributing deployments × 3492 timestamps  →  3104 QC-passing samples

pooled mean fCO2 = 417.5 uatm  (range 281.9–792.7)
  11SS20240501    BE-SOOP-Simon Stevin            n=   28  mean=287.1 uatm
  11SS20240601    BE-SOOP-Simon Stevin            n= 2382  mean=381.3 uatm
  11SS20240801    BE-SOOP-Simon Stevin            n=  276  mean=643.1 uatm
  11SS20241001    BE-SOOP-Simon Stevin            n=  128  mean=520.6 uatm
  11SS20241101    BE-SOOP-Simon Stevin            n=  268  mean=461.6 uatm
  11SS20241201    BE-SOOP-Simon Stevin            n=   22  mean=541.3 uatm


## Same query via the data-passport proxy (port 8080)

Reads the stores over HTTP through `run_proxy.py`. Each `.values` access
records chunk fetches; on session close a passport is minted.

The proxy is assumed to run at https://zarr.icos-cp.eu

In [12]:
import json
from datapassport_zarr import open_zarr

OBSPACK_URL = "https://zarr.icos-cp.eu/icos-obspack.zarr"
FLUXNET_URL = "https://zarr.icos-cp.eu/icos-fluxnet.zarr"

# Same xarray expressions as above — over HTTP, with passports.
# .load() forces the lat/lon coord chunks to fetch eagerly while keeping
# them as xarray DataArrays (with dims), so .where(..., drop=True) accepts
# the boolean mask. Without the eager fetch, lazy reads of lat/lon
# mid-mask can flake on slow / SSH-tunneled connections.
# record_query() captures bbox + surviving stations explicitly so the
# passport JSON shows them.

bbox = {"lat": [LAT_MIN, LAT_MAX], "lon": [LON_MIN, LON_MAX]}

print("=== Obspack — CO2 ===")
with open_zarr(OBSPACK_URL, group="co2", verbose=True) as ds_co2:
    ds_co2.record_query({"bbox": bbox, "time": {"time_co2": [T0, T1]}})
    ds_co2._ds["lat"].load()
    ds_co2._ds["lon"].load()
    co2_nl = (
        ds_co2["co2"]
              .where(
                  (ds_co2.lat >= LAT_MIN) & (ds_co2.lat <= LAT_MAX) &
                  (ds_co2.lon >= LON_MIN) & (ds_co2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time_co2=slice(T0, T1))
              .compute()
    )
    ds_co2.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in co2_nl.station.values
    ]})
    print(f"Obspack CO2 — {len(co2_nl.station)} stations × {len(co2_nl.time_co2)} timestamps")
    print(f"  mean across NL+2024 = {float(co2_nl.mean()):.2f} ppm")

obspack_passport = ds_co2._passport

=== Obspack — CO2 ===
Obspack CO2 — 8 stations × 8784 timestamps
  mean across NL+2024 = 434.44 ppm
[datapassport_zarr] Session closed (84 chunks) — Handle/CP not configured, no PID minted.


In [13]:
print("\n=== Fluxnet — monthly NEE ===")
with open_zarr(FLUXNET_URL, group="_combined/fluxnet_mm", verbose=True) as ds_nee:
    ds_nee.record_query({"bbox": bbox, "time": {"time": [T0, T1]},
                         "select": {"ustar_threshold": "VUT", "nee_variant": "REF"}})
    ds_nee._ds["lat"].load()
    ds_nee._ds["lon"].load()
    nee_nl = (
        ds_nee["NEE"]
              .sel(ustar_threshold="VUT", nee_variant="REF")
              .where(
                  (ds_nee.lat >= LAT_MIN) & (ds_nee.lat <= LAT_MAX) &
                  (ds_nee.lon >= LON_MIN) & (ds_nee.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    ds_nee.record_query({"surviving_stations": [
        s.decode() if isinstance(s, (bytes, bytearray)) else str(s)
        for s in nee_nl.station.values
    ]})
    print(f"Fluxnet NEE — {len(nee_nl.station)} stations × {len(nee_nl.time)} months")
    print(f"  mean across NL+2024 = {float(nee_nl.mean()):.3f} {ds_nee['NEE'].attrs.get('units','')}")

fluxnet_passport = ds_nee._passport

print("\n──── Obspack passport ─────────────────────────────────────────────────")
print(json.dumps(obspack_passport, indent=2, default=str))

print("\n──── Fluxnet passport ─────────────────────────────────────────────────")
print(json.dumps(fluxnet_passport, indent=2, default=str))

import pathlib
saved = sorted(pathlib.Path(".passport").glob("*.json"))[-2:] if pathlib.Path(".passport").exists() else []
if saved:
    print(f"\nSaved passport files: {[str(p) for p in saved]}")


=== Fluxnet — monthly NEE ===
Fluxnet NEE — 4 stations × 12 months
  mean across NL+2024 = -0.413 umol m-2 s-1
[datapassport_zarr] Session closed (25 chunks) — Handle/CP not configured, no PID minted.

──── Obspack passport ─────────────────────────────────────────────────
{
  "passport_pid": "",
  "passport_url": "",
  "chunks": 84,
  "bytes_served": 27007966,
  "arrays": [
    "altitude",
    "calibration_scale",
    "citation",
    "co2",
    "co2_qc_flag",
    "country",
    "dataset_name",
    "intake_height",
    "lat",
    "lon",
    "site_name",
    "source_doi",
    "station",
    "station_url",
    "time_co2"
  ],
  "queries": [
    {
      "op": "user_recorded",
      "group": "co2",
      "bbox": {
        "lat": [
          50.7,
          53.6
        ],
        "lon": [
          3.3,
          7.3
        ]
      },
      "time": {
        "time_co2": [
          "2024-01-01",
          "2024-12-31"
        ]
      }
    },
    {
      "variable": "co2",
      "group":

### SOCAT via the proxy

With the `_combined` group in place, SOCAT becomes a single xarray
expression — same pattern as Obspack and Fluxnet above.

In [7]:
SOCAT_URL = "https://zarr.icos-cp.eu/icos-socat.zarr"

print("=== SOCAT — fCO2 ===")
with open_zarr(SOCAT_URL, group="_combined", verbose=True) as ds_fco2:
    ds_fco2.record_query({"bbox": bbox, "time": {"time": [T0, T1]}})
    ds_fco2._ds["lat"].load()
    ds_fco2._ds["lon"].load()
    fco2_nl = (
        ds_fco2["fCO2"]
              .where(
                  (ds_fco2.lat >= LAT_MIN) & (ds_fco2.lat <= LAT_MAX) &
                  (ds_fco2.lon >= LON_MIN) & (ds_fco2.lon <= LON_MAX),
                  drop=True,
              )
              .sel(time=slice(T0, T1))
              .compute()
    )
    valid = fco2_nl.values[np.isfinite(fco2_nl.values)]
    surviving = [str(d) for d in fco2_nl.deployment.values]
    ds_fco2.record_query({"surviving_deployments": surviving,
                          "qc_passing_samples":   int(valid.size)})
    print(f"SOCAT fCO2 — {len(surviving)} deployments × "
          f"{fco2_nl.sizes.get('time', 0)} timestamps  →  "
          f"{valid.size} QC-passing samples")
    if valid.size:
        print(f"  pooled mean fCO2 = {valid.mean():.1f} µatm")

socat_passport = ds_fco2._passport

print("\n──── SOCAT passport ──────────────────────────────────────────────────")
print(json.dumps(socat_passport, indent=2, default=str))

=== SOCAT — fCO2 ===
SOCAT fCO2 — 7 deployments × 3492 timestamps  →  3104 QC-passing samples
  pooled mean fCO2 = 417.5 µatm
[datapassport_zarr] Session closed (69 chunks) — Handle/CP not configured, no PID minted.

──── SOCAT passport ──────────────────────────────────────────────────
{
  "passport_pid": "",
  "passport_url": "",
  "chunks": 69,
  "bytes_served": 20202674,
  "arrays": [
    "citation",
    "country_code",
    "deployment",
    "fCO2",
    "fixed",
    "lat",
    "lon",
    "platform_name",
    "source_doi",
    "station_id",
    "time"
  ],
  "queries": [
    {
      "op": "user_recorded",
      "group": "_combined",
      "bbox": {
        "lat": [
          50.7,
          53.6
        ],
        "lon": [
          3.3,
          7.3
        ]
      },
      "time": {
        "time": [
          "2024-01-01",
          "2024-12-31"
        ]
      }
    },
    {
      "variable": "fCO2",
      "group": "_combined"
    },
    {
      "op": "where",
      "variable":